Imports


In [4]:
import pandas as pd
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

Load Data

In [5]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


Preprocess Data

In [7]:
initial_review_count = len(df)
df = df.dropna(subset=['Date','Source','Summary'])
# Robust date parsing: try pandas, fallback to dateutil for hard cases
def _parse_dates(series):
    parsed = pd.to_datetime(series, errors='coerce')
    if parsed.isna().any():
        from dateutil import parser
        mask = parsed.isna()
        def _try(x):
            try:
                return parser.parse(x, dayfirst=False)
            except Exception:
                return pd.NaT
        parsed.loc[mask] = series[mask].apply(lambda x: _try(x) if isinstance(x, str) else pd.NaT)
    return pd.to_datetime(parsed, errors='coerce')

df['Date_parsed'] = _parse_dates(df['Date'].astype(str))
n_missing = int(df['Date_parsed'].isna().sum())
if n_missing:
    print(f"Warning: {n_missing} rows have unparseable dates and will be dropped.")
df = df.dropna(subset=['Date_parsed']).reset_index(drop=True)

# Normalize to ISO date string (YYYY-MM-DD) and create timestamps used by BERTopic
df['Date'] = df['Date_parsed'].dt.strftime('%Y-%m-%d')
docs = df["Summary"].tolist()
timestamps = pd.to_datetime(df["Date"]).dt.date

# 1. Preprocessing (Required for LDA)
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_lda(text):
    text = re.sub(r'\s+', ' ', text).lower() # Remove extra whitespace & lowercase
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words and len(w) > 3]
    return tokens

# Process your existing 'docs' list
processed_docs = [preprocess_lda(doc) for doc in docs]

# 2. Create Dictionary and Corpus
id2word = corpora.Dictionary(processed_docs)
# Filter out words that appear in less than 5 docs or more than 50% of docs
id2word.filter_extremes(no_below=5, no_above=0.5) 
corpus = [id2word.doc2bow(text) for text in processed_docs]

/home/codespace/.local/lib/python3.12/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
/home/codespace/.local/lib/python3.12/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...


Train LDA Model

In [8]:
# 3. Train LDA Model
# Note: You MUST pre-define the number of topics (k)
num_topics = 12 
lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, 
                     random_state=42, passes=10, alpha='auto')

View Topics

In [9]:
# 4. View Topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic: {idx} \nWords: {topic}")

Topic: 0 
Words: 0.055*"federal" + 0.044*"reserve" + 0.039*"credit" + 0.037*"facility" + 0.029*"board" + 0.027*"rate" + 0.026*"market" + 0.023*"fund" + 0.021*"primary" + 0.020*"announces"
Topic: 1 
Words: 0.061*"federal" + 0.052*"reserve" + 0.046*"treasury" + 0.040*"release" + 0.039*"oversight" + 0.037*"congressional" + 0.036*"commission" + 0.030*"financial" + 0.029*"report," + 0.028*"recent"
Topic: 2 
Words: 0.070*"billion" + 0.047*"quarter" + 0.041*"loss" + 0.025*"fannie" + 0.024*"fdic" + 0.024*"announces" + 0.022*"2009." + 0.022*"fourth" + 0.019*"asset" + 0.018*"report"
Topic: 3 
Words: 0.040*"state" + 0.027*"covid-19" + 0.022*"united" + 0.020*"order" + 0.017*"individual" + 0.017*"announces" + 0.015*"governor" + 0.015*"first" + 0.014*"president" + 0.014*"public"
Topic: 4 
Words: 0.101*"u.s." + 0.076*"purchase" + 0.074*"treasury" + 0.054*"preferred" + 0.042*"stock" + 0.041*"capital" + 0.039*"department" + 0.037*"bank" + 0.033*"billion" + 0.029*"program."
Topic: 5 
Words: 0.031*"econo